# 02 — Model training and SHAP analysis

Train the full pipeline (LightGBM + CQR + hierarchical residuals), then walk through a representative SHAP analysis for a 1955 J-45 in Excellent original condition vs the same guitar refinished — quantifying the refin penalty as the model sees it.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from gibson_price.models.train import train_full
from gibson_price.models.predict import PredictionRequest, predict_price

artifact = train_full()
print(f'Test MAPE: {artifact.test_mape:.1%}')
print(f'80% PI coverage: {artifact.coverage_80:.1%}')

In [ ]:
# A 1955 J-45 Excellent original
clean = predict_price(PredictionRequest(
    brand='Gibson', model_family='J-45', year=1955, condition_grade=6,
    has_original_case=True,
))
print(f'Original: ${clean.median_usd:,.0f}  PI [${clean.interval_low_usd:,.0f}, ${clean.interval_high_usd:,.0f}]')
print(clean.natural_language_summary)

# Same guitar, refinished
refin = predict_price(PredictionRequest(
    brand='Gibson', model_family='J-45', year=1955, condition_grade=6,
    refinished=True, has_original_case=True,
))
print(f'Refinished: ${refin.median_usd:,.0f}  PI [${refin.interval_low_usd:,.0f}, ${refin.interval_high_usd:,.0f}]')
print(f'Refin penalty: {(refin.median_usd / clean.median_usd - 1) * 100:.0f}%')

In [ ]:
import pandas as pd
pd.DataFrame([{'feature': c.feature, 'value': c.value, 'contribution_usd': c.contribution_usd} for c in clean.top_contributors])